# ⚙️ Módulos y Librerías: Cierre de Ficheros y Gestión de Recursos (`close`, `try-finally`, `with`)

| Propiedad | Detalle |
| :--- | :--- |
| 📍 **Módulo** | Módulos, Paquetes y Librerías / Manejo de Ficheros |
| 🎯 **Objetivo** | Comprender la importancia del método `close()`, prevenir la fuga de descriptores de archivos (*resource leaks*), garantizar la descarga de buffers de memoria en disco y utilizar patrones robustos como `try-finally` y Administradores de Contexto (`with`). |
| 🛠️ **Conceptos** | `close()`, `closed`, `Buffer Flush`, `Resource Leakage`, `try-finally`, `Context Managers (with)` |

---

### 💡 La Importancia de Cerrar Ficheros (`close()`)

Cuando se abre un archivo en Python, el sistema operativo asigna un descriptor de archivo (*file descriptor*) y reserva memoria temporal (buffer). Ejecutar **`file.close()`** es fundamental por tres razones críticas:

1. **Liberación de Recursos:** Evita alcanzar el límite máximo de archivos abiertos simultáneamente por el sistema operativo.
2. **Integridad de Datos:** Fuerza la escritura inmediata de los datos pendientes almacenados en el buffer de la memoria RAM hacia el disco físico.
3. **Desbloqueo de Archivos:** Permite que otros procesos u operaciones del sistema accedan o modifiquen el archivo sin bloqueos de lectura/escritura.

---

### 🗂️ Estrategias para la Liberación de Recursos

| Estrategia | Ventajas | Desventajas | Uso Recomendado |
| :--- | :--- | :--- | :--- |
| 🔌 **Cierre Manual (`close()`)** | Sencillo y directo. | Propenso a fallar si ocurre un error (*exception*) antes de llegar a la línea del `close()`. | Scripts muy simples de una sola pasada. |
| 🛡️ **Garantizado (`try-finally`)** | Garantiza que `close()` se ejecute incluso si ocurren errores intermedios. | Requiere un bloque de código más extenso y explícito. | Cuando se maneja manualmente la instancia del archivo. |
| ✨ **Context Manager (`with`)** | Cierra el archivo **automáticamente** al salir del bloque, siendo limpio y pythonico. | Ninguna relevante. | **Estándar recomendado de la industria.** |

In [ ]:
import os
from typing import Any

# 🛰️ [1] Banner visual de inicio de ejecución
print("=" * 80)
print("🐍 MANEJO DE FICHEROS | Cierre Explícito y Gestión de Recursos con close()")
print("=" * 80)

# ================================================================================
# 🧪 1. DEMOSTRACIÓN DE CIERRE MANUAL Y VERIFICACIÓN DE ATRIBUTO CLOSED
# ================================================================================
# Uso directo de close() y verificación del estado del stream.
# ================================================================================

print("\n--- 🚀 [PASO 1]: Cierre Manual de Ficheros ---")

sample_file_path: str = "demo_closing_example.txt"

# 🔌 [1] Abrimos un archivo con la función open()
file_stream = open(sample_file_path, "w", encoding="utf-8")

# 🔌 [2] Realizamos operaciones en el fichero
file_stream.write("Escritura de datos de prueba para la lección de cierre de ficheros.\n")

# Verificación del estado antes de cerrar
print(f"📋 ¿El archivo está cerrado antes de llamar a close()?: {file_stream.closed}")

# 🔌 [3] Cerramos el fichero para liberar recursos del sistema
file_stream.close()

# Verificación del estado tras el cierre
print(f"  ✅ ¿El archivo está cerrado tras llamar a close()?: {file_stream.closed}")

print("-" * 65)

# ⚠️ Intento de operación sobre un archivo cerrado (Lanza ValueError)
try:
    file_stream.write("Intento de escribir en un archivo ya cerrado.")
except ValueError as error:
    print("  ❌ Captura de Error: No se puede operar sobre un archivo cerrado.")
    print(f"  🔍 Detalle técnico: {error}")

# Limpieza del archivo temporal
if os.path.exists(sample_file_path):
    os.remove(sample_file_path)

## 🧱 Patrones Robustos: `try-finally` vs Administrador de Contexto (`with`)

### 💡 La Propiedad `closed`:
Todo objeto de archivo dispone del atributo booleano **`file.closed`**, el cual devuelve `True` si el recurso ya ha sido liberado de la memoria.

Para evitar que un error intermedio impida la ejecución de `close()`, se utiliza el patrón `try-finally` o se delega la responsabilidad al entorno mediante la cláusula `with`.

In [ ]:
# ================================================================================
# 🧪 2. PATRÓN RESILIENTE TRY-FINALLY Y CONTEXT MANAGERS
# ================================================================================
print("\n--- 🚀 [PASO 2]: Garantía de Cierre con try-finally y statement with ---")

safe_file_path: str = "demo_safe_closing.txt"

# ⚙️ [1] Enfoque 1: Garantía de cierre explícito con try-finally
manual_stream = open(safe_file_path, "w", encoding="utf-8")
try:
    manual_stream.write("Datos procesados con el patrón try-finally.\n")
    print("📋 Operación de escritura completada dentro del bloque try.")
finally:
    # El bloque finally siempre se ejecuta, incluso si ocurre una excepción
    manual_stream.close()
    print(f"  ✅ [try-finally] Archivo cerrado de forma segura. Status closed: {manual_stream.closed}")

print("-" * 65)

# ⚙️ [2] Enfoque 2: Uso del Context Manager con la sentencia 'with' (Recomendado)
with open(safe_file_path, "a", encoding="utf-8") as context_stream:
    context_stream.write("Nuevos datos añadidos mediante Context Manager.\n")
    print("📋 Operación de escritura completada dentro del bloque with.")

# Al salir del bloque 'with', Python cierra automáticamente el archivo
print(f"  ✨ [with statement] Archivo cerrado implícitamente. Status closed: {context_stream.closed}")

# Limpieza del archivo de prueba
if os.path.exists(safe_file_path):
    os.remove(safe_file_path)

## 🛒 Ejercicios Prácticos Avanzados Integrados

---

### 🧪 Ejercicio 1: Monitor de Descriptores y Cierre Garantizado para Procesamiento Masivo

**Escenario Empresarial:** Un pipeline de datos debe procesar múltiples archivos en lote (*batch*). Se debe implementar un gestor que valide el cierre automático de cada recurso procesado para prevenir caídas por agotamiento de descriptores de archivos (*too many open files*) en el sistema operativo.

In [ ]:
# ================================================================================
# 🧪 3. EJERCICIO PRÁCTICO 1: GESTOR Y AUDITOR DE LIBERACIÓN DE RECURSOS
# ================================================================================
print("\n--- 🧪 [PASO 3.1]: Ejercicio 1 - Auditor de Cierre de Ficheros en Lote ---")


def batch_file_processor(file_list: list[str]) -> dict[str, Any]:
    """Procesa una lista de archivos y garantiza el cierre e integridad de cada uno.

    Args:
        file_list (list[str]): Lista de nombres de archivos a crear y procesar.

    Returns:
        dict[str, Any]: Reporte de ejecución con el estado final de cierre de cada archivo.
    """
    audit_results: dict[str, Any] = {
        "processed_files": 0,
        "closed_verification": {}
    }

    for filename in file_list:
        file_handle = None
        try:
            # [1] Apertura del archivo
            file_handle = open(filename, "w", encoding="utf-8")
            file_handle.write(f"Registro en lote para el archivo: {filename}\n")
            audit_results["processed_files"] += 1
        except Exception as error:
            print(f"❌ Error al procesar '{filename}': {error}")
        finally:
            # [2] Cierre garantizado en la cláusula finally
            if file_handle and not file_handle.closed:
                file_handle.close()

            # [3] Registrar verificación de cierre
            if file_handle:
                audit_results["closed_verification"][filename] = file_handle.closed
                print(f"  🔒 Archivo '{filename}' liberado. Estado cerrado: {file_handle.closed}")

    return audit_results


# Creación de lista de archivos para prueba en lote
batch_files: list[str] = ["batch_job_1.tmp", "batch_job_2.tmp", "batch_job_3.tmp"]

print(f"📋 Iniciando procesamiento masivo de {len(batch_files)} archivos...")
execution_audit: dict[str, Any] = batch_file_processor(batch_files)

print("\n📋 Resultado de la auditoría de recursos:")
print(f"  🔹 Archivos procesados con éxito: {execution_audit['processed_files']}")
print(f"  🔹 Confirmación de cierre total:  {all(execution_audit['closed_verification'].values())}")

# Limpieza del lote de archivos de prueba
for temp_file in batch_files:
    if os.path.exists(temp_file):
        os.remove(temp_file)

# --------------------------------------------------------------------------------
# 🏁 [PASO 4]: Cierre de Ejecución
# --------------------------------------------------------------------------------
print("\n" + "=" * 80)
print("✅ [FIN DE EJECUCIÓN] Lección de Cierre de Ficheros finalizada.")
print("=" * 80)